# Group 9: Character Error Rate (CER)

**Course activity · Basic explanation, Python code, and a two-model experiment**

How close is a model's text to the text we expected? This notebook explains one
way to answer that question: **Character Error Rate**, or **CER**.

You do not need experience with CER to follow this notebook. Read each explanation,
then run the code below it. The small examples come before the real model results.

## What you will learn

1. **A — Understand CER:** reference text, predicted text, and three error types.
2. **B — Calculate by hand:** use the formula on two worked examples.
3. **C — Set our rules:** decide exactly what counts as a character difference.
4. **D — Write Python code:** calculate the smallest number of changes.
5. **E — Check the code:** test known answers and compare with JiWER.
6. **F — Compare two LLMs:** score the real answers collected for this activity.
7. **G — Explain the results:** understand CER's strengths and limits.
8. **H — Review and sources:** prepare for the report and live demonstration.

## Before you start

- Select the **Py311-TF** Python kernel in VS Code. A CPU is enough; no GPU or
  TensorFlow code is needed for this activity.
- Run cells from top to bottom. Some later cells use functions created earlier.
- This notebook does **not** contact an AI service or train a model. The actual
  model answers were collected manually and are stored in Section F.
- The only extra scoring library is **JiWER**. Setup instructions appear before
  we use it.
- The code outputs have been cleared so you can generate them yourself.


In [1]:
import jiwer

reference = "kitten"
prediction = "sitting"

cer = jiwer.cer(reference, prediction)

print("Reference:", reference)
print("Prediction:", prediction)
print(f"CER: {cer:.2%}")          # CER: 50.00%

Reference: kitten
Prediction: sitting
CER: 50.00%


## A. What does CER measure?

CER measures differences between two texts **character by character**.

| Term | Basic meaning | Example |
|---|---|---|
| Reference | The expected text | `cat` |
| Prediction, also called hypothesis | The model's text | `cut` |

Here, `a` became `u`: one character changed. The reference contains three
characters, so the error rate is $1/3$, or about **33.33%**.

**Lower CER means a closer text match.** A score of **0%** means the two texts
match exactly under the chosen text rules. It does not prove that the reference
itself is factually correct.

CER is useful when exact wording matters, such as reading text from an image or
turning speech into written text. It does not understand the meaning of an answer.

### Three types of error

We describe errors as changes from the reference to the prediction:

| Type | Letter | Meaning | Reference → prediction |
|---|---|---|---|
| Substitution | S | Replace one character | `cat` → `cut` |
| Deletion | D | A reference character is missing | `cat` → `ct` |
| Insertion | I | An extra character appears | `cat` → `cart` |

Each change costs **1**. Matching characters cost **0**. We choose the comparison
that needs the **smallest total number of changes**, rather than simply counting
characters that differ at the same position. An insertion can shift later letters.


## B. Formula and two worked examples

$$
\mathrm{CER} = \frac{S + D + I}{N}
$$

- **S:** substitutions; **D:** deletions; **I:** insertions.
- **N:** number of characters in the **reference**, including spaces under our rules.
- Multiply the result by 100 to express it as a percentage.

A stored score of `0.4` means **40%**, not 0.4%. Our code uses `:.2%` to display
scores as percentages with two decimal places.

### Example 1 — `plant` → `blat`

| Reference | Prediction | What happened? |
|---|---|---|
| p | b | Substitution |
| l | l | Match |
| a | a | Match |
| n | — | Deletion |
| t | t | Match |

$S=1$, $D=1$, $I=0$, and $N=5$.

$$\mathrm{CER}=\frac{1+1+0}{5}=0.4=40\%$$

### Example 2 — `cat` → `chats`

| Reference | Prediction | What happened? |
|---|---|---|
| c | c | Match |
| — | h | Insertion |
| a | a | Match |
| t | t | Match |
| — | s | Insertion |

$S=0$, $D=0$, $I=2$, and $N=3$.

$$\mathrm{CER}=\frac{0+0+2}{3}\approx0.6667=66.67\%$$

The dash means **no character**, not a literal dash in the text.

### Can CER exceed 100%?

Yes. Reference `a` and prediction `abcd` need three insertions. The reference
length is one, so CER is $3/1=300\%$. CER is an error ratio, not a probability.


## C. Our text comparison rules

The formula alone does not decide how text is prepared. We must state our rules
and use them equally for both models.

| Item | Our rule |
|---|---|
| Capital letters | `C` and `c` are different. |
| Punctuation | Keep and count it. |
| Spaces | Keep all spaces, including spaces at the beginning and end. |
| Line breaks | Keep and count them if they occur in the stored text. |
| Text cleanup | Do not strip spaces, lowercase, or otherwise change text. |
| Empty reference | Reject it with a clear error: the usual formula divides by zero. |
| Empty prediction | Allow it: every reference character is missing. |
| Character unit | Python Unicode code points (numbered text units). |

**A character unit is not always a visible symbol.** In Khmer and other languages,
a visible symbol may contain several code points. Our code does not combine them
into visible-symbol groups or change Unicode forms. This matters when interpreting
scores outside the simple English examples used here. See the Python source in
Section H.

These are valid choices for this experiment, not universal CER requirements.
JiWER removes beginning and ending spaces by default; later we configure it to
keep them so its rules match ours. JiWER also has its own special rules for empty
references; we keep our explicit rejection rule. See the JiWER sources in Section H.


## D. Build the CER calculation in Python

### D1. Check the Python environment

This cell reports the Python used by the selected kernel. It does not install
anything or change the environment. The conda environment should be **Py311-TF**.


In [49]:
import sys
from importlib import metadata

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)

try:
    print("JiWER:", metadata.version("jiwer"))
except metadata.PackageNotFoundError:
    print("JiWER is not installed. See the setup cell in Section E.")


Python: 3.11.15
Interpreter: /home/super/miniconda3/envs/Py311-TF/bin/python
JiWER: 4.0.0


### D2. Look at two text values

A Python string stores text. `len()` counts its code points. `repr()` shows quotes
and makes spaces or empty text easier to see. A variable is a name that holds a value.


In [50]:
example_reference = "plant"
example_prediction = "blat"

print("Reference:", repr(example_reference))
print("Prediction:", repr(example_prediction))
print("Reference length:", len(example_reference))
print("Prediction length:", len(example_prediction))


Reference: 'plant'
Prediction: 'blat'
Reference length: 5
Prediction length: 4


### D3. Find the smallest number of character changes

This number is called **Levenshtein distance (the smallest number of insertions,
deletions, and substitutions needed to match two texts)**.

The function compares small parts of both texts and remembers the cheapest costs.
This is **dynamic programming (solving a larger problem using saved answers to
smaller problems)**.

How to read the code:

1. `previous_row` stores costs from the previous reference character.
2. `current_row` builds costs for the current reference character.
3. For each pair of characters, calculate deletion, insertion, and replacement costs.
4. `min()` keeps the cheapest option. A matching pair adds zero replacement cost.
5. The last value is the minimum total. The function returns the total, not separate
   S, D, and I counts. We will use JiWER to display those counts later.

For long texts this method does more work: it compares each reference character
with each prediction character. Short classroom examples are enough here.


In [51]:
def character_distance(reference, prediction):
    """Return the minimum character edits between two strings."""
    # Creating each prediction prefix from empty text needs insertions.
    previous_row = list(range(len(prediction) + 1))

    for i, reference_char in enumerate(reference, start=1):
        # Removing this reference prefix to make empty text needs i deletions.
        current_row = [i]

        for j, prediction_char in enumerate(prediction, start=1):
            replacement_cost = 0 if reference_char == prediction_char else 1

            deletion = previous_row[j] + 1
            insertion = current_row[j - 1] + 1
            substitution = previous_row[j - 1] + replacement_cost

            current_row.append(min(deletion, insertion, substitution))

        previous_row = current_row

    return previous_row[-1]


example_errors = character_distance(example_reference, example_prediction)
print("Minimum character changes:", example_errors)
print("Expected: 2 — one substitution and one deletion.")


Minimum character changes: 2
Expected: 2 — one substitution and one deletion.


### D4. Turn the error count into CER

The next function checks its inputs, calls our distance function, and divides by
the reference length. `raise` reports an input problem and stops the function.
It does not remove or change text.


In [52]:
def calculate_cer(reference, prediction):
    """Return CER as a ratio; reject non-text inputs and empty references."""
    if not isinstance(reference, str):
        raise TypeError("Reference must be text.")
    if not isinstance(prediction, str):
        raise TypeError("Prediction must be text.")
    if len(reference) == 0:
        raise ValueError("Reference must contain at least one character.")

    errors = character_distance(reference, prediction)
    return errors / len(reference)


example_score = calculate_cer(example_reference, example_prediction)
print("CER as a number:", example_score)
print(f"CER as a percentage: {example_score:.2%}")
print("Expected: 0.4 and 40.00%")


CER as a number: 0.4
CER as a percentage: 40.00%
Expected: 0.4 and 40.00%


## E. Check the calculation

### E1. Test examples with known answers

These are **teaching examples**, not collected model outputs. They check matches,
all three error types, spaces, capitalization, an empty prediction, and CER above
100%. `assert` stops the cell if a result does not match its expected value.


In [53]:
# Each row: reference, prediction, expected minimum error count.
check_cases = [
    ("cat", "cat", 0),
    ("cat", "cut", 1),
    ("cat", "ct", 1),
    ("cat", "cart", 1),
    ("plant", "blat", 2),
    ("cat", "chats", 2),
    ("cat", "", 3),
    ("a", "abcd", 3),
    ("cat", "Cat", 1),
    ("cat", "cat!", 1),
    ("cat", " cat", 1),
    ("cat", "cat ", 1),
    ("a cat", "acat", 1),
]

for ref, pred, expected_errors in check_cases:
    errors = character_distance(ref, pred)
    score = calculate_cer(ref, pred)
    assert errors == expected_errors, (ref, pred, errors, expected_errors)
    assert abs(score - expected_errors / len(ref)) < 1e-12
    print(f"{ref!r} → {pred!r}: errors={errors}, CER={score:.2%} — PASS")

print(f"\nKnown-answer checks passed: {len(check_cases)}/{len(check_cases)}")


'cat' → 'cat': errors=0, CER=0.00% — PASS
'cat' → 'cut': errors=1, CER=33.33% — PASS
'cat' → 'ct': errors=1, CER=33.33% — PASS
'cat' → 'cart': errors=1, CER=33.33% — PASS
'plant' → 'blat': errors=2, CER=40.00% — PASS
'cat' → 'chats': errors=2, CER=66.67% — PASS
'cat' → '': errors=3, CER=100.00% — PASS
'a' → 'abcd': errors=3, CER=300.00% — PASS
'cat' → 'Cat': errors=1, CER=33.33% — PASS
'cat' → 'cat!': errors=1, CER=33.33% — PASS
'cat' → ' cat': errors=1, CER=33.33% — PASS
'cat' → 'cat ': errors=1, CER=33.33% — PASS
'a cat' → 'acat': errors=1, CER=20.00% — PASS

Known-answer checks passed: 13/13


### E2. Check invalid inputs

An empty reference is different from an empty prediction. Our function rejects
an empty reference. It also rejects values such as numbers or `None` instead of text.

`try` attempts a call. `except` catches an expected error so we can show its message.
These messages below are planned input checks, not unexpected notebook failures.


In [54]:
invalid_cases = [
    ("", "cat", ValueError),
    ("", "", ValueError),
    (123, "cat", TypeError),
    ("cat", None, TypeError),
]

for ref, pred, expected_exception in invalid_cases:
    try:
        calculate_cer(ref, pred)
    except expected_exception as error:
        print(f"{ref!r}, {pred!r}: PASS — {error}")
    else:
        raise AssertionError("The function should have rejected this input.")


'', 'cat': PASS — Reference must contain at least one character.
'', '': PASS — Reference must contain at least one character.
123, 'cat': PASS — Reference must be text.
'cat', None: PASS — Prediction must be text.


### E3. Check against JiWER

JiWER is an existing Python library that calculates CER and other text error
measures. Comparing it with our function helps catch mistakes.

If the environment cell reported that JiWER is missing, uncomment the installation
line below and run this cell. Otherwise, leave the installation line commented.
The example installation uses JiWER 4.0.0. The cell prints your actual version.

We do not use JiWER's default whitespace cleanup. Our custom transformation
(preparation rule) only splits the original text into characters.


In [55]:
# Run this line only if JiWER is missing from the selected kernel.
# %pip install jiwer==4.0.0

import jiwer

print("JiWER version:", metadata.version("jiwer"))

# No Strip(), lowercase conversion, or punctuation removal.
character_rules = jiwer.Compose([
    jiwer.ReduceToListOfListOfChars(),
])


def inspect_character_errors(reference, prediction):
    """Return JiWER counts using the same input rules as our CER function."""
    calculate_cer(reference, prediction)  # Apply our input checks first.
    return jiwer.process_characters(
        reference,
        prediction,
        reference_transform=character_rules,
        hypothesis_transform=character_rules,
    )


for ref, pred, expected_errors in check_cases:
    result = inspect_character_errors(ref, pred)
    our_score = calculate_cer(ref, pred)
    library_errors = result.substitutions + result.deletions + result.insertions
    assert library_errors == expected_errors
    assert abs(our_score - result.cer) < 1e-12

print(f"JiWER comparison checks passed: {len(check_cases)}/{len(check_cases)}")


JiWER version: 4.0.0
JiWER comparison checks passed: 13/13


### E4. Display the three error counts

For our first worked example, expect **S=1, D=1, I=0**, and **40% CER**.
For our second, expect **S=0, D=0, I=2**, and about **66.67% CER**.

Some text pairs have more than one cheapest comparison. Their separate S, D,
and I counts can differ between valid comparisons, while the minimum total and
CER stay the same. These two simple examples have the counts shown below.


In [56]:
for ref, pred in [("plant", "blat"), ("cat", "chats")]:
    result = inspect_character_errors(ref, pred)
    print(f"\nReference: {ref!r} | Prediction: {pred!r}")
    print("Substitutions:", result.substitutions)
    print("Deletions:", result.deletions)
    print("Insertions:", result.insertions)
    print("Reference characters:", len(ref))
    print(f"CER: {result.cer:.2%}")



Reference: 'plant' | Prediction: 'blat'
Substitutions: 1
Deletions: 1
Insertions: 0
Reference characters: 5
CER: 40.00%

Reference: 'cat' | Prediction: 'chats'
Substitutions: 0
Deletions: 0
Insertions: 2
Reference characters: 3
CER: 66.67%


## F. Compare real answers from two LLMs

### F1. What we tested

**Task:** correct spelling while keeping the wording, capitalization, and punctuation
unchanged except where needed to fix spelling. Return only the corrected sentence.

We collected the answers manually in the chat services. We did not train models
or make API calls (requests sent by a program).

| Service | Label reported by the student | What we know |
|---|---|---|
| Claude | Sonnet 5.5 Medium | Recorded as reported; the exact model version was not independently checked. |
| Gemini | Auto | An automatic selection mode; a specific model was not recorded. |

These labels describe the services/settings tested. In particular, **Auto is not
an exact model version**, so repeating this experiment later may use a different model.
The exact collection date and chat settings were not recorded in the notebook.
Do not invent those details; add them only if the original chat records confirm them.

### Experiment procedure

1. Use the same prompt and input for both services.
2. Use a separate new chat for each task, as instructed during collection.
   Chat-history evidence is not stored here, so that part of the procedure is not independently verified.
3. Do not include the reference answer in the prompt.
4. Copy the actual answer without correcting it or intentionally removing extra text.
5. Score both answers with the same character rules.

The stored answers below are the **text reported by the student**. The notebook
cannot verify invisible characters or formatting that may have changed while copying.

Five tasks were prepared, but only **Tasks 1 and 2** were tested. Tasks 3–5 have
no collected answers and receive no scores. We stopped after the first two tasks;
this is a small demonstration, not a broad ranking of the services.


In [57]:
# Prepared inputs and references. These are not model predictions.
evaluation_tasks = [
    {"id": 1, "input": "The cat is sleping.",
     "reference": "The cat is sleeping."},
    {"id": 2, "input": "I drink watter every day.",
     "reference": "I drink water every day."},
    {"id": 3, "input": "She goes to schol.",
     "reference": "She goes to school."},
    {"id": 4, "input": "Please close the dor.",
     "reference": "Please close the door."},
    {"id": 5, "input": "We are learning Pythn.",
     "reference": "We are learning Python."},
]

tasks_by_id = {task["id"]: task for task in evaluation_tasks}
assert len(tasks_by_id) == len(evaluation_tasks), "Task IDs must be unique."

prompt_instruction = (
    "Correct the spelling mistakes in the sentence below. "
    "Keep the wording, capitalization, and punctuation unchanged "
    "except where needed to fix spelling. "
    "Return only the corrected sentence, without explanations."
)

for task in evaluation_tasks:
    task["prompt"] = prompt_instruction + "\n\n" + task["input"]

# Show the prompts for the two tested tasks, without their references.
for task_id in [1, 2]:
    print(f"\n=== PROMPT FOR TASK {task_id} ===")
    print(tasks_by_id[task_id]["prompt"])



=== PROMPT FOR TASK 1 ===
Correct the spelling mistakes in the sentence below. Keep the wording, capitalization, and punctuation unchanged except where needed to fix spelling. Return only the corrected sentence, without explanations.

The cat is sleping.

=== PROMPT FOR TASK 2 ===
Correct the spelling mistakes in the sentence below. Keep the wording, capitalization, and punctuation unchanged except where needed to fix spelling. Return only the corrected sentence, without explanations.

I drink watter every day.


### F2. Store the collected answers

The four records below are the actual answers reported during our activity:
two tasks × two services. They are not generated by this cell.

The cell creates a fresh list each time it runs, so running it again does not
add extra copies. Keep these answers unchanged when repeating the original experiment.


In [58]:
model_results = [
    {"task_id": 1, "model": "Claude — Sonnet 5.5 Medium",
     "answer": "The cat is sleeping."},
    {"task_id": 1, "model": "Gemini — Auto",
     "answer": "The cat is sleeping."},
    {"task_id": 2, "model": "Claude — Sonnet 5.5 Medium",
     "answer": "I drink water every day."},
    {"task_id": 2, "model": "Gemini — Auto",
     "answer": "I drink water every day."},
]

# Check that no answer is missing, unknown, or recorded twice.
expected_models = {"Claude — Sonnet 5.5 Medium", "Gemini — Auto"}
expected_pairs = {
    (model_name, task_id)
    for model_name in expected_models
    for task_id in [1, 2]
}
recorded_pairs = {(row["model"], row["task_id"]) for row in model_results}
assert len(recorded_pairs) == len(model_results), "Duplicate answer records found."
assert recorded_pairs == expected_pairs, "Missing or unexpected answer records."

for row in model_results:
    assert isinstance(row["answer"], str), "Each answer must be text."
    print(f"Task {row['task_id']} | {row['model']} | {row['answer']!r}")

print("\nCollected answers:", len(model_results))
print("Tested tasks: 1 and 2. Untested tasks: 3, 4, and 5.")


Task 1 | Claude — Sonnet 5.5 Medium | 'The cat is sleeping.'
Task 1 | Gemini — Auto | 'The cat is sleeping.'
Task 2 | Claude — Sonnet 5.5 Medium | 'I drink water every day.'
Task 2 | Gemini — Auto | 'I drink water every day.'

Collected answers: 4
Tested tasks: 1 and 2. Untested tasks: 3, 4, and 5.


### F3. Calculate each answer's score

We create a new results list from the original records. Each row contains the
reference, actual answer, three error counts, reference length, and CER.

`display()` shows a Markdown table inside the notebook. This needs no pandas library.


In [59]:
from IPython.display import Markdown, display

scored_results = []

for row in model_results:
    ref = tasks_by_id[row["task_id"]]["reference"]
    pred = row["answer"]
    details = inspect_character_errors(ref, pred)
    score = calculate_cer(ref, pred)
    assert abs(score - details.cer) < 1e-12

    scored_results.append({
        **row,
        "reference": ref,
        "substitutions": details.substitutions,
        "deletions": details.deletions,
        "insertions": details.insertions,
        "errors": details.substitutions + details.deletions + details.insertions,
        "reference_characters": len(ref),
        "cer": score,
    })

# Show the actual texts before the score table.
for row in scored_results:
    print(f"\nTask {row['task_id']} — {row['model']}")
    print("Reference:", repr(row["reference"]))
    print("Answer:   ", repr(row["answer"]))

lines = [
    "| Task | Model | S | D | I | Reference characters | CER |",
    "|---|---|---|---|---|---|---|",
]
for row in scored_results:
    lines.append(
        f"| {row['task_id']} | {row['model']} | "
        f"{row['substitutions']} | {row['deletions']} | "
        f"{row['insertions']} | {row['reference_characters']} | "
        f"{row['cer']:.2%} |"
    )

display(Markdown("\n".join(lines)))



Task 1 — Claude — Sonnet 5.5 Medium
Reference: 'The cat is sleeping.'
Answer:    'The cat is sleeping.'

Task 1 — Gemini — Auto
Reference: 'The cat is sleeping.'
Answer:    'The cat is sleeping.'

Task 2 — Claude — Sonnet 5.5 Medium
Reference: 'I drink water every day.'
Answer:    'I drink water every day.'

Task 2 — Gemini — Auto
Reference: 'I drink water every day.'
Answer:    'I drink water every day.'


| Task | Model | S | D | I | Reference characters | CER |
|---|---|---|---|---|---|---|
| 1 | Claude — Sonnet 5.5 Medium | 0 | 0 | 0 | 20 | 0.00% |
| 1 | Gemini — Auto | 0 | 0 | 0 | 20 | 0.00% |
| 2 | Claude — Sonnet 5.5 Medium | 0 | 0 | 0 | 24 | 0.00% |
| 2 | Gemini — Auto | 0 | 0 | 0 | 24 | 0.00% |

### F4. Calculate overall CER for each service

For an overall score, add the errors and reference lengths **before dividing**:

$$
\mathrm{Overall\ CER}=\frac{\sum(S+D+I)}{\sum N}
$$

This is not generally the same as the plain average of task percentages. Longer
references contribute more characters to this overall score. Both services must
be evaluated on the same tasks for this comparison.

Task 1 has **20** reference characters and Task 2 has **24**, including spaces
and punctuation. Each service therefore has **44** reference characters in total.


In [60]:
model_summary = {}

for row in scored_results:
    model_name = row["model"]
    if model_name not in model_summary:
        model_summary[model_name] = {
            "tasks": 0,
            "errors": 0,
            "reference_characters": 0,
        }

    totals = model_summary[model_name]
    totals["tasks"] += 1
    totals["errors"] += row["errors"]
    totals["reference_characters"] += row["reference_characters"]

lines = [
    "| Model | Tested tasks | Total errors | Reference characters | Overall CER |",
    "|---|---|---|---|---|",
]
for model_name, totals in sorted(model_summary.items()):
    totals["cer"] = totals["errors"] / totals["reference_characters"]
    lines.append(
        f"| {model_name} | {totals['tasks']} | {totals['errors']} | "
        f"{totals['reference_characters']} | {totals['cer']:.2%} |"
    )

display(Markdown("\n".join(lines)))

# Confirm the expected result for the stored experiment.
for totals in model_summary.values():
    assert totals["tasks"] == 2
    assert totals["reference_characters"] == 44
    assert totals["errors"] == 0
    assert totals["cer"] == 0
print("Stored experiment checks passed.")


| Model | Tested tasks | Total errors | Reference characters | Overall CER |
|---|---|---|---|---|
| Claude — Sonnet 5.5 Medium | 2 | 0 | 44 | 0.00% |
| Gemini — Auto | 2 | 0 | 44 | 0.00% |

Stored experiment checks passed.


### F5. What did we learn?

Both services achieved **0% CER on the two tested spelling-correction tasks**.
Their reported answers exactly matched the prepared references under our rules.
The comparison found no score difference on these tasks.

This does **not** mean that either service is perfect, that they are equally good
at all tasks, or that the untested tasks would also receive 0%. The examples were
short and simple. Only two tasks, with one reported answer per service per task,
were tested. The exact Gemini model was not identified, and collection metadata
is incomplete. Wider claims would need more varied tasks and better records.

We stopped collecting after two tasks. **Tasks 3–5 remain untested.** We do not
make up answers or include them in the totals.


## G. Strengths, limits, and conclusion

### G1. Strengths

- **Easy to explain:** count character edits and divide by reference length.
- **Sensitive to small text errors:** one missing letter or extra punctuation affects the score.
- **No word splitting required:** the calculation uses character units.
- **Useful for exact text tasks:** for example, checking text read from an image
  or written from speech.

### G2. Limits

- **It does not understand meaning.** A correct answer with different wording
  may receive a high CER.
- **A small CER can hide an important factual mistake.** Replacing one digit can
  change an age, amount, date, or duration.
- **The reference must be reliable.** A wrong reference can make a correct answer
  appear wrong.
- **Text rules affect scores.** Case, whitespace, punctuation, and Unicode handling
  must be described. Scores from different rules are not directly comparable.
- **The character unit matters.** A Python code point is not always one symbol
  a person sees, especially for text with combining marks.
- **It is not a universal LLM quality score.** CER does not measure reasoning,
  helpfulness, safety, or factual correctness by itself.

### G3. Demonstrate a factual mistake with a low CER

The next example is a **made-up teaching example**, not an additional model result.
Only one digit is wrong, yet the duration changes. This illustrates a limitation
of CER; it does not make any claim about a real product or service.


In [61]:
fact_reference = "The card is valid for 4 years."
fact_prediction = "The card is valid for 5 years."

print("Reference:", fact_reference)
print("Prediction:", fact_prediction)
print("Character errors:", character_distance(fact_reference, fact_prediction))
print(f"CER: {calculate_cer(fact_reference, fact_prediction):.2%}")
print("The text is close, but the stated number of years is wrong.")


Reference: The card is valid for 4 years.
Prediction: The card is valid for 5 years.
Character errors: 1
CER: 3.33%
The text is close, but the stated number of years is wrong.


### G4. Conclusion

CER measures the smallest number of character substitutions, deletions, and
insertions relative to the reference length. Lower CER means a closer character
match. A score can exceed 100% when the prediction contains many extra characters.

Our Python implementation agrees with known answers and is checked against JiWER
using the same text rules. In the collected experiment, both services matched both
references exactly and achieved an overall CER of **0% across 44 reference
characters per service**. This small experiment does not establish a general winner.

Use CER when exact text matters. For open-ended Q&A, review meaning and factual
correctness separately. A fluent answer, a repeatable answer, and a close text
match are different things.


## H. Review, demonstration, and sources

### Live demonstration plan

1. Explain reference, prediction, and S/D/I using `plant` → `blat`.
2. Show its hand calculation: $2/5=40\%$.
3. Run the distance and CER functions and show the same result.
4. Show the known-answer checks and JiWER comparison checks.
5. Show the four collected answers and both overall scores.
6. Explain why a tie on two simple tasks does not rank the services generally.
7. Show the one-digit teaching example to explain why CER cannot check facts.

### Before submitting

- Restart the **Py311-TF** kernel and run all cells in order. If JiWER is missing,
  run the installation line in Section E, then run all cells again.
- Confirm all checks pass, the comparison has four answer records, and each
  service has two tasks, zero errors, and 44 reference characters.
- Save the notebook with its new outputs using **Ctrl+S**.
- Add group member names and IDs to the report and slides.
- Add confirmed collection dates/model details if the original chat records provide them.
- Prepare the required **PDF report, Python source code, result tables, slides,
  and live demonstration**. This notebook provides the content and results;
  those separate submission files still need to be prepared.

### Sources

1. **Teacher's assignment:**
   [Activity: LLM Performance Evaluation](../Activity_12A-Nov-Ch3-ADV_Programming_Friday.pdf).
   Group 9 is assigned CER. This is the source for the activity requirements.
2. **JiWER: character processing and error counts:**
   [Official API documentation](https://jitsi.github.io/jiwer/reference/process/).
   Describes character edit distance, CER, and substitution/deletion/insertion counts.
3. **JiWER: default text preparation:**
   [Official transformation documentation](https://jitsi.github.io/jiwer/reference/transformations/).
   Documents removal of beginning and ending whitespace by default. We override
   this preparation so JiWER matches our exact-text rules.
4. **JiWER: empty-reference behavior:**
   [Official project README](https://github.com/jitsi/jiwer#a-note-on-empty-references).
   JiWER 4.0 introduced special handling of empty references. Our wrapper instead
   rejects empty references, as stated in Section C.
5. **Python: Unicode strings and code points:**
   [Official Unicode HOWTO](https://docs.python.org/3/howto/unicode.html).
   Supports the explanation of the text units used by Python strings.

The spelling-correction answers are student-reported observations, not claims
from these documents. The hand calculations and known-answer cases are teaching
examples created for this notebook.
